### Import Committee Reports from Congress API

In [26]:
# Inital Setup
import numpy as np
import pandas as pd
import requests
import json
import dotenv
import os
import yaml
import llm
import pprint

from lxml import etree
from lxml import html
from bs4 import BeautifulSoup

In [4]:
botname = 'targ'
version = '0.0'
email = 'kve5hd@virginia.edu'
useragent =f'{botname}/{version} ({email}) python-requests/{requests.__version__}'
headers = {'User-Agent':useragent}
headers

{'User-Agent': 'targ/0.0 (kve5hd@virginia.edu) python-requests/2.32.5'}

In [5]:
root = "https://api.congress.gov//v3"
dotenv.load_dotenv()
congresskey = os.getenv('congresskey')

In [6]:
params = {'format': 'json',
          'api_key': congresskey}

In [8]:
endpoint = f'/committee-report'

r = requests.get(root + endpoint,
                 params=params,
                 headers=headers)
r

<Response [200]>

In [9]:
myjson = r.json()

In [10]:
myjson.keys()

dict_keys(['pagination', 'reports', 'request'])

In [11]:
myjson['reports']

[{'chamber': 'House',
  'citation': 'H. Rept. 119-1',
  'cmte_rpt_id': 289187,
  'congress': 119,
  'number': 1,
  'part': 1,
  'type': 'HRPT',
  'updateDate': '2025-05-27T14:12:39Z',
  'url': 'https://api.congress.gov/v3/committee-report/119/HRPT/1?format=json'},
 {'chamber': 'House',
  'citation': 'H. Rept. 119-2',
  'cmte_rpt_id': 289189,
  'congress': 119,
  'number': 2,
  'part': 1,
  'type': 'HRPT',
  'updateDate': '2025-05-27T14:12:41Z',
  'url': 'https://api.congress.gov/v3/committee-report/119/HRPT/2?format=json'},
 {'chamber': 'House',
  'citation': 'H. Rept. 119-3',
  'cmte_rpt_id': 289190,
  'congress': 119,
  'number': 3,
  'part': 1,
  'type': 'HRPT',
  'updateDate': '2025-05-27T14:12:42Z',
  'url': 'https://api.congress.gov/v3/committee-report/119/HRPT/3?format=json'},
 {'chamber': 'House',
  'citation': 'H. Rept. 119-4',
  'cmte_rpt_id': 289191,
  'congress': 119,
  'number': 4,
  'part': 1,
  'type': 'HRPT',
  'updateDate': '2025-05-27T14:13:32Z',
  'url': 'https://api

In [12]:
cmt_reports = pd.json_normalize (myjson, record_path = ['reports'])

In [13]:
cmt_reports.columns

Index(['chamber', 'citation', 'cmte_rpt_id', 'congress', 'number', 'part',
       'type', 'updateDate', 'url'],
      dtype='str')

In [14]:
print(cmt_reports.iloc[0].to_dict())

{'chamber': 'House', 'citation': 'H. Rept. 119-1', 'cmte_rpt_id': 289187, 'congress': 119, 'number': 1, 'part': 1, 'type': 'HRPT', 'updateDate': '2025-05-27T14:12:39Z', 'url': 'https://api.congress.gov/v3/committee-report/119/HRPT/1?format=json'}


In [17]:
row = cmt_reports.iloc[0]
row

chamber                                                    House
citation                                          H. Rept. 119-1
cmte_rpt_id                                               289187
congress                                                     119
number                                                         1
part                                                           1
type                                                        HRPT
updateDate                                  2025-05-27T14:12:39Z
url            https://api.congress.gov/v3/committee-report/1...
Name: 0, dtype: object

In [24]:
r = requests.get(
    f"{root}/{endpoint}/{row['congress']}/{row['type']}/{row['number']}/text",
    headers=headers,
    params=params)

pprint.pprint(r.json())

{'pagination': {'count': 2},
 'request': {'congress': '119',
             'contentType': 'application/json',
             'format': 'json',
             'reportNumber': '1',
             'reportType': 'hrpt'},
 'text': [{'formats': [{'isErrata': 'N',
                        'type': 'Formatted Text',
                        'url': 'https://www.congress.gov/119/crpt/hrpt1/generated/CRPT-119hrpt1.htm'}]},
          {'formats': [{'isErrata': 'N',
                        'type': 'PDF',
                        'url': 'https://www.congress.gov/119/crpt/hrpt1/CRPT-119hrpt1.pdf'}]}]}


In [25]:
htm_url = r.json()['text'][0]['formats'][0]['url']
response = requests.get(htm_url)
response.status_code

200

In [27]:
# Parse the HTML content using BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

# Extract the text
text = soup.get_text()
pprint.pprint(text)

('\n'
 '\n'
 '119th Congress   }                                      {       Report\n'
 '                        HOUSE OF REPRESENTATIVES\n'
 ' 1st Session     }                                      {        119-1\n'
 '\n'
 '======================================================================\n'
 '\n'
 '\n'
 ' \n'
 ' PROVIDING FOR CONSIDERATION OF THE BILL (H.R. 471) TO EXPEDITE UNDER \n'
 '   THE NATIONAL ENVIRONMENTAL POLICY ACT OF 1969 AND IMPROVE FOREST \n'
 'MANAGEMENT ACTIVITIES ON NATIONAL FOREST SYSTEM LANDS, ON PUBLIC LANDS \n'
 'UNDER THE JURISDICTION OF THE BUREAU OF LAND MANAGEMENT, AND ON TRIBAL \n'
 'LANDS TO RETURN RESILIENCE TO OVERGROWN, FIRE-PRONE FORESTED LANDS, AND \n'
 'FOR OTHER PURPOSES, AND PROVIDING FOR CONSIDERATION OF THE BILL (S. 5) \n'
 '  TO REQUIRE THE SECRETARY OF HOMELAND SECURITY TO TAKE INTO CUSTODY \n'
 ' ALIENS WHO HAVE BEEN CHARGED IN THE UNITED STATES WITH THEFT, AND FOR \n'
 '                             OTHER PURPOSES\n'
 '\n'
 '             